# ICL Mechanisms: In-Context Learning Analysis

This notebook demonstrates IRTK's `icl_mechanisms` module for mechanistic
analysis of in-context learning:

1. **Task vector extraction** – the representation shift caused by demonstrations
2. **ICL head identification** – which heads carry demonstration information
3. **Implicit gradient descent** – does attention implement gradient steps?
4. **Label sensitivity** – is the model genuinely learning in-context?
5. **Demonstration order effects** – recency bias and order sensitivity

## Why This Matters

In-context learning (ICL) mechanisms reveal how models learn from examples in the prompt without weight updates. Task vectors, induction head identification, and implicit gradient descent tests probe whether ICL uses the same circuits as few-shot learning or implements a fundamentally different algorithm.

**Key references:**
- [Olsson et al. (2022) "In-context Learning and Induction Heads"](https://arxiv.org/abs/2209.11895) — Discovers induction heads and training phase transitions

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np

from irtk import HookedTransformer, HookedTransformerConfig
from irtk.icl_mechanisms import (
    extract_task_vector,
    icl_head_identification,
    implicit_gradient_descent_test,
    icl_label_sensitivity,
    demonstration_order_effect,
)

In [ ]:
# Build a small model
cfg = HookedTransformerConfig(
    n_layers=2, d_model=16, n_ctx=32, d_head=4, n_heads=4, d_vocab=50,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)

# Simulate demonstrations + query
demo_tokens = jnp.array([0, 1, 2, 3, 4, 5, 6, 7])  # demos + query
query_tokens = jnp.array([6, 7])  # query only
print(f"Demo prompt: {demo_tokens.tolist()}")
print(f"Query only: {query_tokens.tolist()}")

## 1. Task Vector Extraction

The task vector is the difference in representation at the query position
between running with and without demonstrations.

In [ ]:
tv = extract_task_vector(
    model, demo_tokens, query_tokens, "blocks.1.hook_resid_post"
)
print(f"Task vector shape: {tv['task_vector'].shape}")
print(f"Task vector norm: {tv['task_vector_norm']:.4f}")
print(f"Baseline norm: {tv['baseline_norm']:.4f}")
print(f"Relative magnitude: {tv['relative_magnitude']:.4f}")

## 2. ICL Head Identification

Find which attention heads contribute most to in-context learning.

In [ ]:
def metric_fn(logits):
    return float(logits[-1, 0])

heads = icl_head_identification(model, demo_tokens, query_tokens, metric_fn)

print(f"ICL scores shape: {heads['head_icl_scores'].shape}")
print(f"Total ICL effect: {heads['total_icl_effect']:.4f}")
print(f"\nTop ICL heads:")
for layer, head, score in heads['top_icl_heads'][:5]:
    print(f"  L{layer}H{head}: {score:.4f}")
print(f"\nLayer profile: {heads['layer_icl_profile'].round(4)}")

## 3. Implicit Gradient Descent Test

Test whether the attention update aligns with a gradient descent direction.

In [ ]:
for layer in range(model.cfg.n_layers):
    gd = implicit_gradient_descent_test(model, demo_tokens, query_tokens, layer)
    print(f"Layer {layer}: alignment={gd['alignment_score']:.4f}, "
          f"attn_norm={gd['attention_update_norm']:.4f}, "
          f"gd_norm={gd['gd_direction_norm']:.4f}")

## 4. Label Sensitivity

How sensitive is the model to corrupting label positions in the demonstrations?

In [ ]:
sens = icl_label_sensitivity(
    model, demo_tokens, metric_fn,
    corruption_positions=[1, 3, 5],  # hypothetical label positions
    n_corruptions=10,
)
print(f"Clean metric: {sens['clean_metric']:.4f}")
print(f"Mean corrupted: {sens['mean_corrupted_metric']:.4f}")
print(f"Sensitivity score: {sens['sensitivity_score']:.4f}")

## 5. Demonstration Order Effect

Measure variance across different orderings of demonstrations.

In [ ]:
demo_chunks = [jnp.array([0, 1]), jnp.array([2, 3]), jnp.array([4, 5])]
query = jnp.array([6, 7])

order = demonstration_order_effect(model, demo_chunks, query, metric_fn, n_shuffles=20)

print(f"Mean metric: {order['mean_metric']:.4f}")
print(f"Std metric: {order['std_metric']:.4f}")
print(f"Min: {order['min_metric']:.4f}, Max: {order['max_metric']:.4f}")
print(f"Order sensitivity: {order['order_sensitivity']:.4f}")